#Optimize = Merge samll files into larger file

In [0]:
from pyspark.sql.functions import col


data = [
    (i, f"Movie_{i%10}", 2000 + (i % 5), round(float((i % 10) + 5 + (i % 3)/10), 1))
    for i in range(100)
]

columns = ["movie_id", "title", "release_year", "rating"]
df = spark.createDataFrame(data, columns)
df.show(5)


In [0]:
load_path="dbfs:/Volumes/databricks_practice/inputdb/movies_optimize_practice"
(    df.repartition(10)  # creates 10 small files
    .write
    .format("delta")
    .mode("overwrite")
    .save(load_path)
)






In [0]:
spark.sql(f"OPTIMIZE delta.`{load_path}`")


#What Is Z-Ordering?

Z-Ordering is a technique in Delta Lake that co-locates related data in the same file.
It rearranges data inside Parquet files based on the values of one or more columns.

So, when you filter on those columns, Spark can skip entire files, reducing I/O and improving query speed.

In [0]:


data = [
    ("Inception", 2010, 8.8),
    ("Interstellar", 2014, 8.6),
    ("The Dark Knight", 2008, 9.0),
    ("Tenet", 2020, 7.5),
    ("Oppenheimer", 2023, 8.7),
    ("Dune", 2021, 8.0),
    ("Avatar", 2009, 7.8),
    ("The Matrix", 1999, 8.7),
] * 10  # repeat for more rows

df = spark.createDataFrame(data, ["title", "release_year", "rating"])
load_path="dbfs:/Volumes/databricks_practice/inputdb/movies_zorder_demo"

# Write as Delta
df.repartition(4).write.format("delta").mode("overwrite").save(load_path)

spark.read.format("delta").load(load_path).display()

In [0]:

# Optimize and apply Z-Order
spark.sql(f"""
    OPTIMIZE delta.`{load_path}`
    ZORDER BY (rating)
""")


spark.read.format("delta").load(load_path).display()

#Clustering in Delta?

Clustering in Delta organizes data within files based on column(s).

Helps reduce the number of files read for queries with filters on clustering columns.

Unlike partitioning, clustering does not create directories; it reorders data within the files.

Unlike zordering this will take care of file separation automatically

In [0]:

from pyspark.sql.functions import col


data = [
    ("Inception", "2010-07-16", 8.8),
    ("Interstellar", "2014-11-07", 8.6),
    ("Tenet", "2020-08-26", 7.5),
    ("Dunkirk", "2017-07-21", 7.9),
    ("Memento", "2000-09-05", 8.4)
]

columns = ["title", "release_date", "rating"]

df = spark.createDataFrame(data, columns)
df.show()

df.createTempView("moview_vw")


In [0]:
%sql
CREATE TABLE databricks_practice.inputdb.tblmovies
USING DELTA
CLUSTERED BY (rating) INTO 2 BUCKETS
AS
SELECT * FROM moview_vw;

In [0]:
#CLuserting in files:
data = [
    ("Inception", "2010-07-16", 8.8),
    ("Interstellar", "2014-11-07", 8.6),
    ("Tenet", "2020-08-26", 7.5),
    ("Dunkirk", "2017-07-21", 7.9),
    ("Memento", "2000-09-05", 8.4)
]

columns = ["title", "release_date", "rating"]
df = spark.createDataFrame(data, columns)

#load_path="dbfs:/Volumes/databricks_practice/inputdb/movies_cluster_demo"
# Write Delta table with bucketing
df.write.format("delta").bucketBy(2, "rating").sortBy("rating").mode("overwrite").saveAsTable("databricks_practice.inputdb.tblmovies")


#Vaccum:
Delta keeps history and old files for time travel and ACID transactions.

Over time, these old files consume storage.

VACUUM removes files that are no longer needed.

Default retention: 7 days. You can reduce it, but not below 1 day for safety.

Syntax:

VACUUM delta.`<path>` [RETAIN <hours> HOURS];


Notes:

Only removes files no longer needed for versioning.

Be careful if you plan to use time travel; vacuumed versions cannot be queried.

In [0]:

data = [
    ("Inception", 8.8),
    ("Interstellar", 8.6),
    ("Tenet", 7.5)
]

columns = ["title", "rating"]
df = spark.createDataFrame(data, columns)

load_path="dbfs:/Volumes/databricks_practice/inputdb/movies_cluster_demo"
# Write initial version
df.write.format("delta").mode("overwrite").save(load_path)


In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, load_path)

# Add a new movie (creates new version)
new_data = [("Dunkirk", 7.9)]
new_df = spark.createDataFrame(new_data, ["title", "rating"])

delta_table.alias("t").merge(
    new_df.alias("s"),
    "t.title = s.title"
).whenNotMatchedInsertAll().execute()


In [0]:
%sql

SELECT * FROM delta.`dbfs:/Volumes/databricks_practice/inputdb/movies_cluster_demo` ;

In [0]:
from delta.tables import DeltaTable
spark.conf.set(
    "spark.databricks.delta.retentionDurationCheck.enabled",
    "false"
)
delta_table = DeltaTable.forPath(spark, "dbfs:/Volumes/databricks_practice/inputdb/movies_cluster_demo")
delta_table.vacuum(1) 



In [0]:
%sql

SELECT * FROM delta.`dbfs:/Volumes/databricks_practice/inputdb/movies_cluster_demo` ;